# Layer 18 PLS scores

Downloads `layer_out/18` at prompt position **-1**, fits a **6-component PLS regression** against
`log10_time_horizon_months`, and writes the scores to CSV with **every prompt-metadata field
preserved** alongside them.

Layer 18 is the choice the redundancy analysis landed on: it is the argmax of held-out target R²
at position -1 in both full sweeps (0.950 and 0.955 mean across datasets), and the per-dataset best
in three of four datasets.

Output per dataset, one row per prompt:

| Columns | What they hold |
|---|---|
| every flattened prompt-metadata field | `task`, `template_id`, `base_value`, `base_unit`, `template_metadata.*`, `task_metadata.*`, … |
| `dataset`, `sample_index`, `batch_file` | provenance back to the cached batch |
| `time_horizon_months`, `log10_time_horizon_months` | the target, in months and as fitted |
| `pls_1` … `pls_6` | the PLS scores |
| `pls_prediction`, `pls_residual` | the model's prediction of the target and its error |

The fitted projection is saved as well, so the identical transform can be applied to new
activations later without refitting.

Both `LAYER` and `POSITION` also accept a list - `LAYER = [17, 18]`, `POSITION = [-2, -1]`. Every
layer-position pair is then concatenated along the feature axis before the PLS fit, laid out
layer-major and position-minor, so a prompt is described by `2,560 x len(LAYER) x len(POSITION)`
floats and every component draws on all of them at once. Output filenames carry the layers and the
positions joined by underscores (`..._layer17_18_pos-2_-1_pls6.csv`).

Activations are staged to an on-disk memory-mapped array (2,560 floats per prompt per layer-position
pair), so a dataset
of any size can be scored without holding it in RAM; batches are downloaded, consumed, and deleted
one group at a time.


## 1. Setup

For a fresh Colab runtime, uncomment the clone line.


In [ ]:
# !git clone -b dev https://github.com/justinshenk/temporal-manifolds.git
# %cd temporal-manifolds
# !mv -n .env.example .env


In [ ]:
import gc
import json
import math
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from dotenv import load_dotenv
from google.cloud import storage
from numpy.lib.format import open_memmap
from plotly.subplots import make_subplots
from sklearn.cross_decomposition import PLSRegression
from tqdm.auto import tqdm

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
load_dotenv(repo_root / '.env')


## 2. Authenticate to Google Cloud

In Colab, `google.colab.auth.authenticate_user()` installs the Application Default Credentials the
storage client needs; without it the client falls back to the Compute Engine metadata service and
fails with a `RefreshError`. Locally the credentials come from
`gcloud auth application-default login`.


In [ ]:
import google.auth

try:
    from google.colab import auth as colab_auth
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    colab_auth.authenticate_user()
    print('Authenticated through Colab.')

try:
    credentials, detected_project = google.auth.default(
        scopes=['https://www.googleapis.com/auth/devstorage.read_only']
    )
except google.auth.exceptions.DefaultCredentialsError as error:
    raise RuntimeError(
        'No Google Cloud credentials found. In Colab run '
        '`from google.colab import auth; auth.authenticate_user()`; '
        'locally run `gcloud auth application-default login`.'
    ) from error

PROJECT_ID = os.getenv('GCP_PROJECT_ID') or detected_project or 'temporal-interp-exp'
print(f'Credentials: {type(credentials).__name__}')
print(f'Project: {PROJECT_ID}')


## 3. Configure

Layer 18 lives in the `expanded_` caches (layers 17-35), so `RANGE_TAG` points there. Naming several
layers in `LAYER` only works for layers held in that same cache folder, since one batch file is the
only thing read.

`task_only` is absent from `DATASETS` because its prompts declare no time horizon - the target
would be undefined for every row.

`FIT_DATASETS` chooses which datasets the shared PLS model learns from, independently of which
datasets get scored. Leave it `None` to fit on all of them; name a subset to fit there and score the
rest as unseen data - for example `FIT_DATASETS = ['conversational_no_output_format']` fits on the
conversational phrasing and reports how that axis transfers to `abstract` and the `plain_*` sets. It
applies only when `SHARED_MODEL` is true; per-dataset models always fit on their own dataset.

**On cost:** each cached batch file holds all nineteen layers at both positions (~25 MB) even
though only one layer at one position is used, so scoring a dataset in full means downloading its
whole folder. `conversational` is 2,618 files (~65 GB); its no-output-format twin covers the same
prompts. Set `BATCH_SAMPLE_SIZE` to score a spread-out subset instead of everything, and
`MAX_ROWS_PER_BATCH` to thin each file.


In [ ]:
BUCKET_NAME = 'temporal-research-bucket'
RANGE_TAG = 'expanded_'                 # Folder tag holding layers 17-35.
LAYER: int | list[int] = 18             # Layer(s) to score; a list concatenates them.
POSITION: int | list[int] = -1          # Prompt token(s) to score; a list concatenates them.
PLS_COMPONENTS = 6


@dataclass(frozen=True)
class Dataset:
    name: str
    gcs_suffix: str


DATASETS = [
    Dataset('conversational_no_output_format', 'NOF_selected_acts'),
    Dataset('abstract', 'abstract_selected_acts'),
    Dataset('plain_english', 'plain_english_selected_acts'),
    Dataset('plain_long', 'plain_long_selected_acts'),
    # Dataset('conversational', 'selected_acts'),   # ~65 GB; the NOF twin covers the same prompts.
]

DATA_DIR = repo_root / 'data' / 'layer18_pls'
OUTPUT_DIR = repo_root / 'results' / 'layer18_pls'

BATCH_SAMPLE_SIZE: int | None = None    # Batch files to score, evenly spaced; None takes every file.
MAX_ROWS_PER_BATCH: int | None = None   # Rows kept per file; None keeps all 128.
DOWNLOAD_WORKERS = 8
FIT_SAMPLE_SIZE = 50_000                # Rows the PLS model is fitted on; None fits on everything.
SHARED_MODEL = True                     # One model across datasets, so scores are comparable.
FIT_DATASETS: list[str] | None = None   # Dataset names the shared model learns from; None uses all.
INCLUDE_PROMPTS = False                 # Write the full prompt text into the CSV.
FEATURE_CHUNK_ROWS = 4_096              # Rows per chunk when transforming from the memmap.
RANDOM_SEED = 0

LAYERS = [LAYER] if isinstance(LAYER, int) else list(LAYER)
if not LAYERS:
    raise ValueError('LAYER must name at least one layer.')
if len(set(LAYERS)) != len(LAYERS):
    raise ValueError(f'LAYER repeats a layer: {LAYERS}')
LAYER_COMPONENTS = [f'layer_out/{layer}' for layer in LAYERS]
LAYER_TAG = '_'.join(str(layer) for layer in LAYERS)

POSITIONS = [POSITION] if isinstance(POSITION, int) else list(POSITION)
if not POSITIONS:
    raise ValueError('POSITION must name at least one prompt token.')
if len(set(POSITIONS)) != len(POSITIONS):
    raise ValueError(f'POSITION repeats a token: {POSITIONS}')
POSITION_TAG = '_'.join(str(position) for position in POSITIONS)
DATASET_NAMES = [dataset.name for dataset in DATASETS]
unknown_fit_datasets = sorted(set(FIT_DATASETS or []) - set(DATASET_NAMES))
if unknown_fit_datasets:
    raise ValueError(
        f'FIT_DATASETS names datasets that are not being scored: {unknown_fit_datasets}. '
        f'Available: {DATASET_NAMES}'
    )
FIT_DATASET_NAMES = list(FIT_DATASETS) if FIT_DATASETS else DATASET_NAMES

rng = np.random.default_rng(RANDOM_SEED)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
client = storage.Client(project=PROJECT_ID, credentials=credentials)
layer_label = (
    LAYER_COMPONENTS[0] if len(LAYERS) == 1 else f'layer_out {LAYERS} concatenated'
)
position_label = (
    f'position {POSITIONS[0]}' if len(POSITIONS) == 1
    else f'positions {POSITIONS} concatenated'
)
print(f'Scoring {layer_label} at {position_label} for {len(DATASETS)} dataset(s).')
if SHARED_MODEL:
    held_out_datasets = [name for name in DATASET_NAMES if name not in FIT_DATASET_NAMES]
    print(f'Shared model fitted on: {FIT_DATASET_NAMES}')
    if held_out_datasets:
        print(f'Scored but not fitted on: {held_out_datasets} (their R2 is an out-of-distribution estimate)')


## 4. Helpers

Horizon conversion matches `scripts/fit_expanded_pls_residual_pca.py` and the
`download_filtered_*` notebooks, so `log10_time_horizon_months` means the same thing everywhere.


In [ ]:
UNIT_TO_MONTHS = {
    'second': 1 / (30.4375 * 86400), 'minute': 1 / (30.4375 * 1440),
    'hour': 1 / (30.4375 * 24), 'day': 1 / 30.4375, 'week': 7 / 30.4375,
    'month': 1.0, 'year': 12.0, 'decade': 120.0, 'century': 1200.0, 'millennium': 12000.0,
}
UNIT_TO_MONTHS.update({f'{unit}s': value for unit, value in list(UNIT_TO_MONTHS.items())})
UNIT_TO_MONTHS['centuries'] = 1200.0
UNIT_TO_MONTHS['millennia'] = 12000.0


def horizon_months(metadata):
    """Return a prompt's horizon in months, or None when it declares none."""
    value = metadata.get('base_value', metadata.get('value'))
    unit = metadata.get('base_unit', metadata.get('unit'))
    if value in (None, 'N/A') or unit in (None, 'N/A'):
        return None
    unit_key = str(unit).lower()
    if unit_key not in UNIT_TO_MONTHS:
        raise ValueError(f'Cannot convert time-horizon unit {unit!r} to months.')
    months = float(value) * UNIT_TO_MONTHS[unit_key]
    return months if months > 0 else None


def flatten_metadata(metadata):
    """Flatten one prompt's nested metadata into dotted scalar fields."""
    flattened = {}
    for key, value in metadata.items():
        if isinstance(value, dict):
            flattened.update({f'{key}.{inner}': item for inner, item in value.items()})
        elif not isinstance(value, (list, tuple, set)):
            flattened[key] = value
    return flattened


def folder_prefix(dataset):
    return f'{RANGE_TAG}{dataset.gcs_suffix}'


def list_batch_blobs(dataset):
    """Every activation batch blob in one dataset folder, ordered by name."""
    prefix = folder_prefix(dataset)
    blobs = sorted(
        (
            blob for blob in client.bucket(BUCKET_NAME).list_blobs(prefix=prefix + '/')
            if Path(blob.name).name.startswith('activations_batch_') and blob.name.endswith('.pt')
        ),
        key=lambda blob: blob.name,
    )
    if not blobs:
        raise FileNotFoundError(f'No activation batches below gs://{BUCKET_NAME}/{prefix}')
    if BATCH_SAMPLE_SIZE is None or len(blobs) <= BATCH_SAMPLE_SIZE:
        return blobs
    picks = np.linspace(0, len(blobs) - 1, BATCH_SAMPLE_SIZE).round().astype(int)
    sampled = [blobs[position] for position in dict.fromkeys(picks.tolist())]
    print(f'  sampling {len(sampled)} of {len(blobs)} batch files, evenly spaced.')
    return sampled


def selected_features(payload):
    """Return this batch's rows for every LAYER_COMPONENTS x POSITIONS pair, concatenated.

    One layer at one position is the plain (rows, hidden) block; otherwise the blocks are laid
    end to end along the feature axis, layer-major and position-minor - so with layers [17, 18]
    and positions [-2, -1] the columns run 17/-2, 17/-1, 18/-2, 18/-1 - giving
    (rows, hidden * len(LAYERS) * len(POSITIONS)).
    """
    activations = payload['activations']
    missing_layers = [name for name in LAYER_COMPONENTS if name not in activations]
    if missing_layers:
        available = sorted(activations)
        raise KeyError(
            f'{missing_layers} not in this batch; it holds {available[0]}..{available[-1]}.'
        )
    cached = list(payload['positions'])
    missing_positions = [position for position in POSITIONS if position not in cached]
    if missing_positions:
        raise ValueError(f'This batch cached positions {cached}; it is missing {missing_positions}.')
    blocks = [
        activations[name][:, cached.index(position), :].to(torch.float32)
        for name in LAYER_COMPONENTS
        for position in POSITIONS
    ]
    return blocks[0] if len(blocks) == 1 else torch.cat(blocks, dim=1)


## 5. Stage one dataset

Batch files are fetched `DOWNLOAD_WORKERS` at a time, the rows for the configured layer(s) at the
configured position(s) are copied - concatenated feature-wise when several are named -
into a memory-mapped `.npy` array, and the `.pt` files are deleted before the next group. Rows whose
prompt declares no horizon are dropped, since the target is undefined for them.


In [ ]:
def stage_dataset(dataset):
    """Download one dataset and return (memmap path, metadata frame, target array)."""
    blobs = list_batch_blobs(dataset)
    download_dir = DATA_DIR / 'batches' / dataset.name
    download_dir.mkdir(parents=True, exist_ok=True)
    features_path = DATA_DIR / f'{dataset.name}_layer{LAYER_TAG}_pos{POSITION_TAG}.npy'

    array = None
    hidden_size = 0
    capacity = 0
    write_cursor = 0
    records = []
    horizons = []
    dropped = 0

    def fetch(blob):
        local_path = download_dir / Path(blob.name).name
        blob.download_to_filename(str(local_path))
        return local_path

    progress = tqdm(total=len(blobs), desc=f'{dataset.name}: staging', leave=False)
    executor = ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS)
    try:
        for group_start in range(0, len(blobs), DOWNLOAD_WORKERS):
            group = blobs[group_start:group_start + DOWNLOAD_WORKERS]
            for local_path in list(executor.map(fetch, group)):
                try:
                    payload = torch.load(local_path, map_location='cpu', weights_only=True, mmap=True)
                    features = selected_features(payload)
                    metadata_rows = payload['prompt_metadata']
                    if len(metadata_rows) != features.shape[0]:
                        raise ValueError(f'{local_path.name}: metadata and activations disagree on rows.')

                    keep, keep_horizons = [], []
                    for offset, metadata in enumerate(metadata_rows):
                        months = horizon_months(metadata)
                        if months is None:
                            dropped += 1
                            continue
                        keep.append(offset)
                        keep_horizons.append(months)
                    if MAX_ROWS_PER_BATCH is not None and len(keep) > MAX_ROWS_PER_BATCH:
                        chosen = np.sort(rng.choice(len(keep), size=MAX_ROWS_PER_BATCH, replace=False))
                        keep = [keep[position] for position in chosen]
                        keep_horizons = [keep_horizons[position] for position in chosen]
                    if not keep:
                        continue

                    if array is None:
                        hidden_size = int(features.shape[1])
                        capacity = len(blobs) * (MAX_ROWS_PER_BATCH or int(features.shape[0]))
                        array = open_memmap(features_path, mode='w+', dtype=np.float32,
                                            shape=(capacity, hidden_size))
                    take = min(len(keep), capacity - write_cursor)
                    if take <= 0:
                        continue
                    keep, keep_horizons = keep[:take], keep_horizons[:take]
                    array[write_cursor:write_cursor + take] = (
                        features[torch.as_tensor(keep, dtype=torch.long)].numpy()
                    )

                    for offset, months in zip(keep, keep_horizons):
                        record = flatten_metadata(metadata_rows[offset])
                        record['dataset'] = dataset.name
                        record['sample_index'] = int(payload['sample_indices'][offset])
                        record['batch_file'] = local_path.name
                        record['time_horizon_months'] = months
                        record['log10_time_horizon_months'] = float(np.log10(months))
                        if INCLUDE_PROMPTS:
                            record['prompt'] = payload['prompts'][offset]
                        records.append(record)
                        horizons.append(months)
                    write_cursor += take
                    del payload, features
                finally:
                    local_path.unlink(missing_ok=True)
                    progress.update(1)
    finally:
        executor.shutdown(wait=True)
        progress.close()
        shutil.rmtree(download_dir, ignore_errors=True)
        if array is not None:
            array.flush()
        del array
        gc.collect()

    if write_cursor == 0:
        raise ValueError(f'{dataset.name}: no rows with a time horizon were staged.')
    if dropped:
        print(f'  dropped {dropped:,} row(s) without a time horizon.')

    metadata_df = pd.DataFrame(records).reset_index(drop=True)
    target = np.log10(np.array(horizons, dtype=np.float64))
    print(f'  staged {write_cursor:,} rows x {hidden_size:,} features -> {features_path.name}')
    return features_path, write_cursor, metadata_df, target


def open_features(features_path, row_count):
    """Open a staged feature array read-only, sliced to the rows actually written."""
    return np.load(features_path, mmap_mode='r')[:row_count]


## 6. Stage every dataset


In [ ]:
staged = {}
for dataset in DATASETS:
    print(f'=== {dataset.name} (gs://{BUCKET_NAME}/{folder_prefix(dataset)}) ===')
    features_path, row_count, metadata_df, target = stage_dataset(dataset)
    staged[dataset.name] = {
        'features_path': features_path, 'row_count': row_count,
        'metadata': metadata_df, 'target': target,
    }

total_rows = sum(entry['row_count'] for entry in staged.values())
print(f'Staged {total_rows:,} rows across {len(staged)} dataset(s).')


## 7. Fit the PLS model

With `SHARED_MODEL = True` one model is fitted on a pooled sample from the datasets named in
`FIT_DATASETS`, so the six scores mean the same thing in every output file and rows can be compared
across datasets. Set `SHARED_MODEL = False` for a per-dataset model instead, in which case
`FIT_DATASETS` does not apply.

The model is fitted on at most `FIT_SAMPLE_SIZE` rows drawn at random and split evenly across the
fitting datasets. R² is reported on the rows held out of the fit; for a dataset excluded from
`FIT_DATASETS` every row is held out, so its R² is a transfer estimate onto phrasing the model never
saw.


In [ ]:
def sample_rows(row_count, budget):
    """Indices of a random subset, or every row when the budget covers it."""
    if budget is None or budget >= row_count:
        return np.arange(row_count)
    return np.sort(rng.choice(row_count, size=budget, replace=False))


def gather_rows(features_path, row_count, indices, chunk=FEATURE_CHUNK_ROWS):
    """Read selected rows out of a staged memmap."""
    features = open_features(features_path, row_count)
    out = np.empty((len(indices), features.shape[1]), dtype=np.float32)
    for start in range(0, len(indices), chunk):
        block = indices[start:start + chunk]
        out[start:start + len(block)] = features[block]
    return out


def fit_pls(feature_blocks, target_blocks):
    """Fit one PLS model on stacked fit samples."""
    features = np.concatenate(feature_blocks, axis=0)
    target = np.concatenate(target_blocks, axis=0)
    model = PLSRegression(n_components=PLS_COMPONENTS, scale=False)
    model.fit(features, target)
    fit_r2 = float(model.score(features, target))
    del features, target
    gc.collect()
    return model, fit_r2


models = {}
fit_indices = {}
if SHARED_MODEL:
    fitting = [name for name in FIT_DATASET_NAMES if name in staged]
    if not fitting:
        raise ValueError(f'None of FIT_DATASETS {FIT_DATASET_NAMES} were staged successfully.')
    per_dataset_budget = None if FIT_SAMPLE_SIZE is None else max(
        PLS_COMPONENTS + 1, FIT_SAMPLE_SIZE // len(fitting)
    )
    blocks, targets = [], []
    for name, entry in staged.items():
        if name not in fitting:
            fit_indices[name] = np.array([], dtype=int)
            print(f'{name}: excluded from the fit; all {entry["row_count"]:,} rows are held out')
            continue
        indices = sample_rows(entry['row_count'], per_dataset_budget)
        fit_indices[name] = indices
        blocks.append(gather_rows(entry['features_path'], entry['row_count'], indices))
        targets.append(entry['target'][indices])
        print(f'{name}: {len(indices):,} rows contributed to the shared fit')
    shared_model, shared_fit_r2 = fit_pls(blocks, targets)
    del blocks, targets
    gc.collect()
    models = {name: shared_model for name in staged}
    print(f'Shared model fitted; R² on the fit sample: {shared_fit_r2:.4f}')
else:
    for name, entry in staged.items():
        indices = sample_rows(entry['row_count'], FIT_SAMPLE_SIZE)
        fit_indices[name] = indices
        block = gather_rows(entry['features_path'], entry['row_count'], indices)
        models[name], fit_r2 = fit_pls([block], [entry['target'][indices]])
        print(f'{name}: fitted on {len(indices):,} rows; R² on the fit sample: {fit_r2:.4f}')
        del block
        gc.collect()


## 8. Score every row and write the CSVs

Scores are computed in chunks straight off the memmap, so the full feature matrix is never
materialized. Each dataset gets its own CSV; a combined file stacks them.


In [ ]:
def transform_in_chunks(model, features_path, row_count, chunk=FEATURE_CHUNK_ROWS):
    """Return (scores, predictions) for every staged row."""
    features = open_features(features_path, row_count)
    scores = np.empty((row_count, PLS_COMPONENTS), dtype=np.float32)
    predictions = np.empty(row_count, dtype=np.float32)
    for start in tqdm(range(0, row_count, chunk), desc='scoring', leave=False):
        block = np.asarray(features[start:start + chunk], dtype=np.float32)
        scores[start:start + len(block)] = model.transform(block).astype(np.float32)
        predictions[start:start + len(block)] = model.predict(block).reshape(-1).astype(np.float32)
    return scores, predictions


score_columns = [f'pls_{index + 1}' for index in range(PLS_COMPONENTS)]
written = []
diagnostics = []
frames = []

for name, entry in staged.items():
    model = models[name]
    scores, predictions = transform_in_chunks(model, entry['features_path'], entry['row_count'])

    frame = entry['metadata'].copy()
    frame[score_columns] = scores
    frame['pls_prediction'] = predictions
    frame['pls_residual'] = frame['log10_time_horizon_months'] - frame['pls_prediction']

    held_out = np.setdiff1d(np.arange(entry['row_count']), fit_indices[name], assume_unique=False)
    def r2(rows):
        if len(rows) < 2:
            return float('nan')
        actual = entry['target'][rows]
        residual = actual - predictions[rows]
        return float(1.0 - np.square(residual).sum() / np.square(actual - actual.mean()).sum())

    csv_path = OUTPUT_DIR / f'{name}_layer{LAYER_TAG}_pos{POSITION_TAG}_pls{PLS_COMPONENTS}.csv'
    frame.to_csv(csv_path, index=False)
    written.append(csv_path)
    frames.append(frame)

    record = {
        'dataset': name,
        'rows': int(entry['row_count']),
        'in_fit_set': bool(len(fit_indices[name])),
        'fit_rows': int(len(fit_indices[name])),
        'held_out_rows': int(len(held_out)),
        'r2_all_rows': r2(np.arange(entry['row_count'])),
        'r2_held_out': r2(held_out),
        'csv': str(csv_path),
    }
    diagnostics.append(record)
    print(f'{name}: {record["rows"]:,} rows -> {csv_path.name} '
          f'(R² all {record["r2_all_rows"]:.4f}, held-out {record["r2_held_out"]:.4f})')
    del scores, predictions
    gc.collect()

combined_path = OUTPUT_DIR / f'all_datasets_layer{LAYER_TAG}_pos{POSITION_TAG}_pls{PLS_COMPONENTS}.csv'
combined = pd.concat(frames, ignore_index=True, sort=False)
combined.to_csv(combined_path, index=False)
print(f'Combined: {len(combined):,} rows -> {combined_path}')
pd.DataFrame(diagnostics).round(4)


## 9. Save the fitted projection

The PLS transform is fully described by the training mean, the rotation matrix, and the regression
coefficients. Saving them means new activations can be scored later without refitting, and without
depending on a pickled scikit-learn version:

```python
loaded = np.load('.../layer18_pls_model.npz')
scores = (new_activations - loaded['x_mean']) @ loaded['x_rotations']
```


In [ ]:
model_path = OUTPUT_DIR / f'layer{LAYER_TAG}_pos{POSITION_TAG}_pls{PLS_COMPONENTS}_model.npz'
reference_model = models[next(iter(models))] if SHARED_MODEL else None

if SHARED_MODEL:
    np.savez(
        model_path,
        x_mean=reference_model._x_mean,
        y_mean=reference_model._y_mean,
        x_rotations=reference_model.x_rotations_,
        x_loadings=reference_model.x_loadings_,
        coef=reference_model.coef_,
        intercept=reference_model.intercept_,
        layer=np.array(LAYERS),
        position=np.array(POSITIONS),
        components=np.array([PLS_COMPONENTS]),
    )
    print(f'Wrote {model_path}')
else:
    for name, model in models.items():
        path = OUTPUT_DIR / f'{name}_layer{LAYER_TAG}_pos{POSITION_TAG}_pls{PLS_COMPONENTS}_model.npz'
        np.savez(path, x_mean=model._x_mean, y_mean=model._y_mean,
                 x_rotations=model.x_rotations_, x_loadings=model.x_loadings_,
                 coef=model.coef_, intercept=model.intercept_,
                 layer=np.array(LAYERS), position=np.array(POSITIONS),
                 components=np.array([PLS_COMPONENTS]))
        print(f'Wrote {path}')

metadata_path = OUTPUT_DIR / f'layer{LAYER_TAG}_pos{POSITION_TAG}_pls{PLS_COMPONENTS}_diagnostics.json'
metadata_path.write_text(json.dumps({
    'layer_components': LAYER_COMPONENTS,
    'positions': POSITIONS,
    'pls_components': PLS_COMPONENTS,
    'shared_model': SHARED_MODEL,
    'fit_datasets': FIT_DATASET_NAMES if SHARED_MODEL else 'per-dataset',
    'bucket': BUCKET_NAME,
    'folders': [folder_prefix(dataset) for dataset in DATASETS],
    'batch_sample_size': BATCH_SAMPLE_SIZE,
    'max_rows_per_batch': MAX_ROWS_PER_BATCH,
    'fit_sample_size': FIT_SAMPLE_SIZE,
    'random_seed': RANDOM_SEED,
    'datasets': diagnostics,
}, indent=2))
print(f'Wrote {metadata_path}')


## 10. Sanity check

The first PLS component should track the target closely. Each dataset gets its own panel, sharing
one axis pair, with a sample of points drawn to keep the figure light.


In [ ]:
PLOT_SAMPLE = 4_000
INK, MUTED, GRID, MARK = '#0b0b0b', '#52514e', '#e6e5e1', '#2a78d6'

names = list(staged)
columns = min(len(names), 3)
rows = int(np.ceil(len(names) / columns))
fig = make_subplots(rows=rows, cols=columns, subplot_titles=names,
                    horizontal_spacing=0.08, vertical_spacing=0.16)
for index, name in enumerate(names):
    frame = frames[index]
    sample = frame.sample(min(PLOT_SAMPLE, len(frame)), random_state=RANDOM_SEED)
    fig.add_trace(
        go.Scattergl(
            x=sample['log10_time_horizon_months'], y=sample['pls_1'],
            mode='markers', marker={'size': 4, 'color': MARK, 'opacity': 0.45},
            name=name, showlegend=False,
            hovertemplate='log10 months %{x:.2f}<br>PLS-1 %{y:.2f}<extra></extra>',
        ),
        row=index // columns + 1, col=index % columns + 1,
    )
fig.update_layout(
    title={'text': f'PLS component 1 against the target<br>'
                   f'<sup>{layer_label} at {position_label}; '
                   f'{PLOT_SAMPLE:,} points sampled per dataset</sup>',
           'font': {'size': 17, 'color': INK}, 'x': 0, 'xanchor': 'left'},
    template='simple_white', height=320 * rows + 120,
    margin={'l': 70, 'r': 30, 't': 100, 'b': 60},
    font={'color': MUTED, 'size': 12},
)
fig.update_xaxes(title_text='log10 time horizon (months)', showgrid=False, linecolor=GRID)
fig.update_yaxes(title_text='PLS-1', gridcolor=GRID, linecolor=GRID)
for annotation in fig.layout.annotations:
    annotation.font.size = 12
    annotation.font.color = INK
fig.show()

print('Correlation of PLS-1 with the target:')
for name, frame in zip(names, frames):
    print(f'  {name:34s} r = {frame["pls_1"].corr(frame["log10_time_horizon_months"]):+.4f}')


## 11. Clean up the staged features

The memmaps are kept so the notebook can be re-run - a different component count, a per-dataset
model - without downloading again. Run this cell when finished.


In [ ]:
# shutil.rmtree(DATA_DIR, ignore_errors=True)
# print(f'Removed {DATA_DIR}')
